### Vector Store Embedding Visualization

This notebook provides an exploratory view into the semantic structure of embeddings stored in the vector database. Its purpose is to make the latent space produced by the embedding model observable, allowing qualitative inspection of how documents are positioned relative to one another after being encoded and persisted.

To enable visualization, high-dimensional embedding vectors are projected into two and three dimensions using `t-SNE`. These projections are not intended to preserve exact distances, but to reveal local structure and clustering tendencies in the data. Each point corresponds to a stored document chunk, enriched with hover metadata that exposes its document type and a short text preview, making it possible to connect spatial patterns with semantic meaning


### Import packages

In [1]:
import itertools
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from sklearn.manifold import TSNE

In [2]:
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.bootstrap import bootstrap

### Load and Set Up Vector Store

All configuration is loaded through the `bootstrap` function.  
The resulting settings are used to initialize the Chroma vector store.  
The vector store connects to a persisted collection on disk and is ready for downstream retrieval or analysis tasks.
After initialization, the underlying collection is retrieved from the vector store. 

In [4]:
# Load all application settings via bootstrap
app_context = await bootstrap()

# Get vectorstore from settings
vectorstore = app_context.vectorstore

INFO     HTTP Request: GET http://localhost:8010/api/v2/auth/identity "HTTP/1.1 200 OK"             ]8;id=8729147;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=8729148;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\

INFO     HTTP Request: GET http://localhost:8010/api/v2/tenants/default_tenant "HTTP/1.1 200 OK"    ]8;id=8729153;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=8729154;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\

INFO     HTTP Request: GET                                                                          ]8;id=8729159;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=8729160;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
         http://localhost:8010/api/v2/tenants/default_tenant/databases/default_database "HTTP/1.1                  
         200 OK"                                                                                                   

INFO     HTTP Request: POST                                                                         ]8;id=8729165;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=8729166;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
         http://localhost:8010/api/v2/tenants/default_tenant/databases/default_database/collections                
         "HTTP/1.1 200 OK"                                                                                         

INFO     Persistence initialized                                                                     ]8;id=8729173;file://E:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\src\rag\persistence\factory.py\factory.py]8;;\:]8;id=8729174;file://E:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\src\rag\persistence\factory.py#104\104]8;;\

INFO     PersistenceContext initialized                                                                  ]8;id=8729181;file://E:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\src\rag\persistence\base.py\base.py]8;;\:]8;id=8729182;file://E:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\src\rag\persistence\base.py#24\24]8;;\

In [6]:
# Retrieve all stored data from the collection, including vectors, raw documents, and metadata
result = vectorstore.get_all_data()

# Convert embeddings to a NumPy array for numerical processing and visualization
vectors = np.array(result["embeddings"])

# Extract original documents associated with each embedding
documents = result["documents"]

# Extract metadata for each document (e.g., document type, source, IDs)
metadatas = result["metadatas"]

# Collect document types from metadata to support grouping and coloring
doc_types = [metadata["topic"] for metadata in metadatas]

# Assign a color to each vector based on its document type
unique_types = sorted(set(doc_types))
palette = itertools.cycle(["blue", "green", "red", "orange", "purple", "brown"])

DOC_TYPE_COLORS = dict(zip(unique_types, palette))
colors = [DOC_TYPE_COLORS[t] for t in doc_types]

INFO     HTTP Request: POST                                                                         ]8;id=8729193;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=8729194;file://e:\Continue\projects\99.pet_projects\02.RAG_FROM_MYSQL\04.final_version\agent\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
         http://localhost:8010/api/v2/tenants/default_tenant/databases/default_database/collections                
         /4e4457ce-5733-4eaf-9eaa-6d5360223e43/get "HTTP/1.1 200 OK"                                               

### Visualize Embeddings with t-SNE

This cell performs a 2D visualization of the vector store embeddings:

1. **Dimensionality reduction**: High-dimensional vectors are projected to 2D using t-SNE for easier visualization.
2. **Color coding**: Each document is assigned a color based on its type.
3. **Interactive scatter plot**: Hover over points to inspect the document type and a snippet of its content.


In [7]:
# Reduce high-dimensional embeddings to 2D for visualization
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create an interactive scatter plot using Plotly
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),  # Color by document type
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],  # Hover info
            hoverinfo="text",
        )
    ]
)

# Configure layout: title, axis labels, size, and margins
fig.update_layout(
    title=f"Vector Store Visualization. Embedding dimension: {len(vectors[0])}",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

### 3D Visualization of Embeddings with t-SNE

This cell projects high-dimensional embeddings into **3D space** using t-SNE and visualizes them interactively:

- **Dimensionality reduction**: Reduces vectors to 3 components for 3D visualization.
- **Color coding**: Each document is assigned a color based on its type.
- **Interactive 3D scatter plot**: Hover over points to see the document type and a snippet of the text.


In [8]:
# Reduce high-dimensional embeddings to 3D for visualization
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create an interactive 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],  # Hover text shows document type and first 100 characters of document
            hoverinfo="text",
        )
    ]
)

# Configure layout: title, axis labels, size, and margins
fig.update_layout(
    title="3D Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)


fig.show()